## Note to self

### Slack
There is a slack for the group. It's located here: https://app.slack.com/client/T9VBHRPSA/C9UF0M6TE
I don't thinkt here is a PW. You just sometimes have to verify it's you by recieving a code by email and entering it. (Slack is connected to the email catharina.hock@stud.uni-heidelberg.de I think.)

### Group wiki
There is a group wiki. It's here: https://gitlab.com/planets-are-cool/wiki/-/wikis/home
username: catharinahock, email: catharina.hock@stud.uni-heidelberg.de, PW: Kekse-0

## Activating VPN
run

`gtk-launch com.cisco.secureclient.gui`

in the command line.

## BinAC
Connection to BinAC is via the command line. Type: `ssh hd_cu284@login.binac2.uni-tuebingen.de`
PW: Schoko etc.

Connecting to BinAC might only work from inside the uni network. Run **gtk-launch com.cisco.secureclient.gui** in the command line to open the cisco client and connect to the VPN. Connecting should then work.

There is a tutorial site here: https://training.bwhpc.de/ilias.php?baseClass=illmpresentationgui&cmd=layout&ref_id=275&obj_id=1



### Uploading files to BinAC

Switch to the folder where you want to copy files in the command line. Then run
`sftp hd_cu284@login.binac2.uni-tuebingen.de:dir_where_you_want_to_put_the_stuff`

Then:
`sftp> put test.txt`

This only works for files though, not whole directories. You need to turn them into a .tar file first :(

There is another way to upload files and even whole folders:

To upload the local file test.txt into the folder test:

`scp test.text hd_cu284@login.binac2.uni-tuebingen.de:~/test/`

To upload the local folder test_dir into the folder test:

`scp -r test_dir hd_cu284@login.binac2.uni-tuebingen.de:~/test/`

Or to upload the local folder test into a workspace:

`scp -r test hd_cu284@login.binac2.uni-tuebingen.de:/pfs/10/work/hd_cu284-work`


## Creating and Managing Workspaces on BinAC

Directly after login, you land in the directory $HOME. This is for permanent storage of source codes etc. and is backed up regularly. It doesn't have a lot of space though. There's also a temporary storage $TMP, but I'm not sure yet for what you would use it.
Then there are workspaces. Those are for storing large amounts of data and performing computations. They're note backed up.

There are several commands to create/view/ etc. workspaces:

| Command                      | Action                                                                                                       |
|------------------------------|--------------------------------------------------------------------------------------------------------------|
| `ws_allocate mywork 30`      | Allocate a work space named **"mywork"** for 30 days.                                                        |
| `ws_allocate myotherwork`    | Allocate a work space named **"myotherwork"** with maximum lifetime. (This is not true, more like lifetime of 1 day -_-)                                        |
| `ws_list -a`                 | List all your work spaces.                                                                                   |
| `ws_find mywork`             | Get absolute path of work space **"mywork"**.                                                                |
| `ws_extend mywork 30`        | Extend lifetime of work space **"mywork"** by 30 days from now. (Not needed, workspaces on BinAC are not limited). |
| `ws_release mywork`          | Manually erase your work space **"mywork"**. Please remove directory content first.                          |


To switch to a workspace, you have to find the absolute path with `ws_find mywork` and then run `cd absolute/path`. To get back to `$HOME`, run `cd $HOME`


## Managing permissions
If you run `ls -l` you can see the permissions for the workspaces:

`[hd_cu284@login02 ~]$ ls -l`

`total 4`

`drwxr-xr-x. 2 hd_cu284 hd_hd 4096 Oct 27 14:30 work`

d means it's a directory (for files it's just -), wxr =  write, execute, read, this are the permissions of the user. xr= exectute, read: this are the permissions of the group (hd_hd). x = execute: this is the permission of everyone else.
If you want to make it private, run `chmod 700 work`.
Then you'll have

`drwx------. 2 hd_cu284 hd_hd 4096 Oct 27 14:30 work`

## Submitting a Job on BinAC

To submit a job, you must first create a `.slurm` file that contains all the commands you would have called in the cmd line if you were to run your simulation on your own computer. It also needs a header that tells slurm (a scheduler) how much memory/time/resources your job needs. You can also tell it where to store outputs (the output you would get normally from running your commands in the cmd line). A long list of the keywords can be found here: https://www.uibk.ac.at/zid/systeme/hpc-systeme/common/tutorials/slurm-tutorial.html

A `.slurm`file can look like this, for example:

`#!/bin/sh`

`#SBATCH --ntasks=1`

`#SBATCH --time=10:00`

`#SBATCH --mem=5000m`

`#SBATCH --job-name=simple`

`echo "Scratch directory: $TMPDIR"`

`echo "Date:"`

`date`

`echo "My job is running on node:"`

`hostname`

`uname -a`

`echo "Some random text" > randomtext.txt`

`sleep 240`

This file basically just outputs some text into the cmd line and creates a file called randomtext.txt with the content "Some random text". It's also possible to run python scripts though!

To submit the job, run 

`sbatch job_name.slurm`

To check on the status of th job, run `sstat {insert job ID here}`. The job ID will be printed upon submitting the job.
You can also cancel a job by running `scancel {inser job ID here}`.


Running a job will produce an output file `output_{Job ID}.txt` that is easiest to access with vim. Just type `vim {Job ID}.out` to see its contents. To quit, type `:q` 
Some jobs might produce error files, too. Those are called `error_{Job ID}.txt`.

## Make SETUP on BinAC
Before running fargo for the first time or whenever you want to switch the setup, you have to run `make SETUP="fargo"`. If you attempt this directly, this will lead to an error because you have to load `mpi` and some other modules first. To do that, type

`module load mpi/openmpi/4.1-gnu-13.3`

`module load devel/cuda/12.6`

More on the module commands and what they do here: https://wiki.bwhpc.de/e/Environment_Modules (I mostly found this out by trial and error.)

If you want to run on GPUs, you also have to run

`make gpu`(This sets GPU=1)
`make parallel`(This sets PARALLEL=1)
`make MPICUDA=1`

Then you can run for example


In [ ]:
!/bin/bash
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=2
#SBATCH --cpus-per-task=2
#SBATCH --mem=32000
#SBATCH --gres=gpu:h200:2
#SBATCH --time=14-00:00:00
#SBATCH --partition=gpu
#SBATCH --mail-type=ALL
#SBATCH --mail-user=cu284@uni-heidelberg.de
#SBATCH --output=%x.out
#SBATCH --error=%x.err
cd $SLURM_SUBMIT_DIR
module purge
module load mpi/openmpi/4.1-gnu-13.3
module load devel/cuda/12.6

mpirun --mca pml ucx --mca btl ^openib --bind-to core --map-by core -report-bindings ./fargo3d -o "outputdir=@outputs/test_with_gpu/" ../test/fargo.par

#Or you can use this to run without GPUs:
./fargo3d -o "outputdir=@outputs/test" ../test/fargo.par


If this creates problems, it's usually because of fargo missing an environment variable to use MPI. In that case, add the following line to the bottom of src/makefile:

`ENVRANK = OMPI_COMM_WORLD_LOCAL_RANK`

I'm not sure what exactly it does, but it's working.

If you want to run without GPUs, remember to run

`make GPU=0 MPICUDA=0`

before switching.

## SLURM example

Léon navigated into his work folder for me to show me an example .slurm file:

In [ ]:
[hd_cu284@login01 hd_cu284-work]$ cd ..
[hd_cu284@login01 work]$ ls
alphafold3                    ho_ariks07-3C_trial                          st_ac129811-ff                        tu_iioba01-amdock
binac_training_2025           ho_ariks07-Ath_CRWNs                         st_ac129811-ofa                       tu_iioba01-expiration-mail-test
db                            ho_ariks07-callus                            st_ac129811-raspa                     tu_iioba01-hlatyping
fr_bd1043-bw24K012            ho_ariks07-pds5                              st_ac129811-spera                     tu_iioba01-MorphoMechanX
fr_cc1057-fr_cc1057-magnolia  ho_ariks07-tomato                            st_ac129811-studies                   tu_iioba01-nextflow-profile
fr_cc1057-magnolia            ho_eihio54-luc                               st_ac131105-h2_pure                   tu_iioba01-sarek
fr_cc1057-spider-monkey       ho_eihio54-wimiq                             st_ac133340-mmom                      tu_iioba01-snakemake
fr_fs1154-anno                ho_ewiho14-Bunting_20250826                  st_ac134849-patrick                   tu_iioba01-snakemake2
fr_fs1154-assembly_gold       ho_ewiho14-ho_ewiho14_angsd_20250324         st_ac134849-transformer_results       tu_iivmi01-bb_halobenzene_learning
fr_fs1154-assembly_plat       ho_faunk93-ddrad_2024                        st_ac136666-ddwsbita                  tu_iivmi01-JNK2_MD
fr_fs1154-assembly_xcln       ho_flawi22-money                             st_ac136666-ddwsbitc                  tu_iivmi01-qm_calcs
fr_fs1154-blob                ho_friox85-ho_friox85-NGS-0                  st_ac137045-sigma                     tu_iizwi01-Staphylococci_SIDs
fr_fs1154-down                ho_graaf20-B6_MAGs                           st_ac139418-raspa_sims                tu_kafdj01-kafdj01
fr_fs1154-ext_gen             ho_graaf20-ChickenPhages                     st_gs112837-Simulations               tu_pelol01-NS_STT
fr_fs1154-hic                 ho_graaf20-Hanna_Test                        tu_bbcmc01-tu_bbcmc01_AF3             tu_zrslv01-apptainer
fr_fs1154-images              ho_graaf20-LH1                               tu_bbcmc01-tu_bbcmc01-binac_training  tu_zrslv01-test
fr_fs1154-repeats             ho_graaf20-LimBiom                           tu_bbcta01-rnaseq                     tu_zxmrn75-milupHPC
fr_mb1783-DMR_2C              ho_graaf20-NaMeco                            tu_bbcta01-singlecell                 tu_zxmzh93-gp_search
fr_mb1783-DMR2C               ho_grulu15-assembly                          tu_bbcta01-tu_bbcta01-rnaseq-0        tu_zxoha56-miluphcuda
fr_ml1168-genomics            ho_grulu15-meta                              tu_bbcta01-tu_bbcta01-singlecell-0    tu_zxoha56-test
fr_rs1092-workspace           ho_hacke-sensoja                             tu_bbvat01-testruns                   tu_zxomw48-mywork
hd_ba282-myotherwork          ho_herrmana-chicken_meta                     tu_bcegj01-scRNA-seq-GJW              tu_zxooq44-test
hd_ba282-mywork               ho_herrmana-LHChicken                        tu_bcelc01-SAM_test                   tu_zxorf45-athena
hd_bu271-KARL                 ho_kersten-bees                              tu_bceql01-tu_bceql01-maize           tu_zxorf45-athena2
hd_cu284-work                 ho_kersten-ho_kersten                        tu_bciqu01-alexa                      tu_zxorz26-artsdb
hd_fp440-edgeon               ho_kezau83-phages                            tu_bciqu01-fldpnn                     tu_zxorz26-artsdb_v2
hd_fp440-rwaur                ho_kezau83-rumen_phages                      tu_bciqu01-interproscan               tu_zxorz26-funannotate2
hd_gx281-BackUp               ho_kezau83-silage                            tu_bciqu01-metaphage                  tu_zxovn09-tu_zxovn09_MA1
hd_lb353-foo                  ho_panke99-ho_panke99                        tu_bciqu01-metaphage_2                tu_zxovu37-myotherwork
hd_lg353-GalFlow              ho_panke99-mywork                            tu_bciqu01-orthofinder                tu_zxovu37-mywork
hd_lh353-NSD0                 ho_ungos18-2025_ophrys                       tu_bciqu01-signalP6                   tu_zxowh45-par
hd_pt254-21cmGalaxy           ho_wilku36-bwhpc-course                      tu_bcoea01-tu_bcoea01                 tu_zxows09-metaphage
hd_pt254-eorflow              ho_wilku36-ho_wilku36                        tu_cpaen01-learning                   tu_zxows09-metaphage2
hd_rv265-ptmp                 ho_yanno24-Limbiom_Lysis                     tu_ctibc01-canu-example-pacbio-trio   tu_zxoyf37-drought_project
hd_su232-dustpy               ho_yogau97-PiFerm2                           tu_ctibc01-canu-example-pacbio-yeast  tu_zxoyf37-phage_project
hd_su232-noad                 ho_yogau97-PiFerm2_metagenomics              tu_ctibc01-hotpool-bench              tu_zxoyf37-smg
hd_tp452-classification       ho_yogau97-PiFerm3                           tu_ctibc01-milupHPC-clone             tu_zxoyf37-smg2
hd_tp452-rnaseq               ho_zurof18-ProBioHuhn                        tu_ctibc01-modflow_pest_debug         tu_zxoyf37-wgd
hd_tp452-segmentation         ho_zurof18-ProBioHuhn2                       tu_ctibc01-phigrape_debug             ul_iuv12-ren_demo
hd_tr323-fargo3d              ka_br6867-mlda_tng                           tu_ctibc01-test                       ul_jav61-demo
hd_tu223-Vlad_work            ka_br6867-tng_mlda                           tu_e16ra01-conda                      ul_jav61-marica_rnaseq
hd_uk280-lognorm_sim          ka_zc3694-c775                               tu_e16ra01-PP_dev                     ul_jav61-pig_single_cell
hd_uk280-test_work            ka_zc3694-icrc25                             tu_e16ra01-pp_dev2_test               ul_jav61-variant
hd_vf176-fargo3d              kn_pop244400-may                             tu_emgsp01-ofwork                     ul_mpf87-BT
hd_vf176-radmc                kn_pop507922-baboon_nuc                      tu_emice01-paleo                      ul_mpf87-DM
hd_vm184-SMBH_dynamics        kn_pop535447-MP                              tu_epabz01-calibration_test           ul_mpf87-GB
hd_wu263-data                 kn_pop537927-kn_pop537927_methods            tu_epacj01-jointInv                   ul_mpf87-ME
ho_aihai07-largebee           kn_pop537936-PAG_WGS                         tu_epajg01-RegioCov                   ul_mpf87-TD
ho_ankis92-BS_seq             kn_pop543547-OTU11                           tu_epird01-GPRI                       ul_pmaity-CO_DBDB_MICE_VESSELS
ho_ankis92-july_2025_ATAC     st_ac112293-202506_SPT                       tu_iijag01-tu_iijag01_par             ul_pmaity-HDF_EC_Coculture_SP100_D2_7_14
ho_ankis92-Meth_intron        st_ac112293-ws_chemistry_transformer_backup  tu_iijcb01-soil                       ul_pmaity-NAIDU_RNASeq
ho_ankis92-tomato_droso       st_ac121527-for5730_p1                       tu_iioba01-alphafold
ho_ankis92-transfer_binac1    st_ac129811-dl_monte                         tu_iioba01-alphafold3
[hd_cu284@login01 work]$ cd ..
[hd_cu284@login01 10]$ ls
project  work
[hd_cu284@login01 10]$ cd project/bw17
bw17b005/ bw17d012/ bw17e006/ bw17g007/ bw17h011/ 
[hd_cu284@login01 10]$ cd project/bw17b005/
-bash: cd: project/bw17b005/: Permission denied
[hd_cu284@login01 10]$ ls
project  work
[hd_cu284@login01 10]$ cd project/bw17d012/
[hd_cu284@login01 bw17d012]$ ls
hd_tr323  hd_vf176
[hd_cu284@login01 bw17d012]$ cd hd_
-bash: cd: hd_: No such file or directory
[hd_cu284@login01 bw17d012]$ cd hd_vf176/
[hd_cu284@login01 hd_vf176]$ ls
4_s_0.5_0_5_2cps_4_orbit.npz           5_b_0.1_8_d1_e-3_small_ld.yml  analysis_plots    old_configs   rho_interp
4_s_0.5_0_5_2cps.yml                   5_b_0.1_8_d1_e-3_small.yml     dhruv             paraview      scripts
5_b_0.1_8_d1_e-3_small_4_orbit.npz     5_s_0.5_0_0.5_e-3_4_orbit.npz  fargo3d_iso       pen_depth     start-jupyter.sh
5_b_0.1_8_d1_e-3_small_ld_4_orbit.npz  5_s_0.5_0_0.5_e-3.yml          jupyter.o1634919  radmc_setups
[hd_cu284@login01 hd_vf176]$ cd fargo3d_iso/
[hd_cu284@login01 fargo3d_iso]$ ls
4_s_0.5_0_5_2cps.o1558067  bin            convert_cloudlet  fargo3d_fs_4_s_0.5_0_5_2cps  Makefile  outputs_SSD  scripts  src  submit_fs.sh       submit.sh   utils
arch                       convert_bondi  convert.sh        in                           outputs   planets      setups   std  submit_restart.sh  test_suite
[hd_cu284@login01 fargo3d_iso]$ cat submit.sh 


The actual .slurm file:

In [ ]:
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=2
#SBATCH --cpus-per-task=2
#SBATCH --mem=32000
#SBATCH --gres=gpu:h200:2
#SBATCH --time=14-00:00:00
#SBATCH --partition=gpu
#SBATCH --mail-type=ALL
#SBATCH --mail-user=huehn@uni-heidelberg.de
#SBATCH --output=%x.o%j
cd $SLURM_SUBMIT_DIR
module purge
module load mpi/openmpi/4.1-gnu-13.3
module load devel/cuda/12.6

mpirun --mca pml ucx --mca btl ^openib --bind-to core --map-by core -report-bindings ./fargo3d_${SLURM_JOB_NAME} -o "outputdir=@outputs_SSD/${SLURM_JOB_NAME}" ./in/${SLURM_JOB_NAME}.par
##./fargo3d_${SLURM_JOB_NAME} -o "outputdir=@outputs_SSD/${SLURM_JOB_NAME}" ./in/${SLURM_JOB_NAME}.par

So, to find Léon's files, starting from my own work folder:
`cd ../../project/bw17d012/`

There are two folders inside. `hd_tr323` is Drishika's and `hd_vf176` is Léon's.

## SLURM tips

To see all jobs you have currently running, type

`squeue -u $USER`

Similarly, to end all jobs you're running, type

`scancel -u $USER`

If you want to submit a JOBNAME (which you can use as ${SLURM_JOB_NAME} inside the slurm file, very useful), you can use

`sbatch --job-name=fargo_a_1e3 run_fargo.slurm`

Always submit the job from inside the fargo3d dir. Otherwise, the script will be unable to find the ./fargo executable.
This is the script I'm using as run_fargo.slurm:

In [ ]:
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=2
#SBATCH --cpus-per-task=2
#SBATCH --mem=32000
#SBATCH --gres=gpu:h200:2
#SBATCH --time=14-00:00:00
#SBATCH --partition=gpu
#SBATCH --mail-type=ALL
#SBATCH --mail-user=cu284@uni-heidelberg.de
#SBATCH --output=%x.out
#SBATCH --error=%x.err
cd $SLURM_SUBMIT_DIR
module purge
module load mpi/openmpi/4.1-gnu-13.3
module load devel/cuda/12.6



mpirun --mca pml ucx --mca btl ^openib --bind-to core --map-by core -report-bindings ./fargo3d -o "outputdir=@outputs/${SLURM_JOB_NAME}" ../binac_runs/${SLURM_JOB_NAME}.par

## Running jupyter notebooks on BinAC
More info can be found here: https://wiki.bwhpc.de/e/BinAC/Software/Jupyterlab


~~First, log in while specifying the port:~~

~~`ssh -D8080 -q hd_cu284@login.binac2.uni-tuebingen.de`~~

~~The Firefox profile I created for remote jupyter notebooks is called BinAC. It can be activated by typing `about:profiles`into the search bar, and clicking on `Launch profile in new browser`.~~

To run jupyter notebooks on BinAC, we habe to submit a job in the form of a slurm file. Thankfully, there are templates for this. I've copied one of them into the work folder. It is called `jupyterlab.slurm`. If for whatever reason the file should be lost, you can copy it easily using

`cp $JUPYTERLAB_EXA_DIR/binac2-minimal-notebook.slurm jupyterlab.slurm`

Now run

`sbatch jupyterlab.slurm`

Now look into the output file. (I tried changing the name of the output file, but for whatever reason this altered the content of the file, too, making it impossible to find the token. So we'll just have to live with slurm creating a new output file with a different name every time we try to start a jupyter notebook :( )
It will look something like this:

In [ ]:
 GNU nano 5.6.1                  slurm-1635475.out                             
Attempt 1: Checked port 18205, port is free ...

   Paste this ssh command in a terminal on local host (i.e., laptop)
   -----------------------------------------------------------------
   ssh -N -L 18205:172.0.0.1:18205 hd_cu284@login.binac2.uni-tuebingen.de

   Open this address in a browser on local host; see token below.
   -----------------------------------------------------------------
   localhost:18205  (prepend with https:// if using a password)

[I 2025-11-04 13:00:03.523 ServerApp] jupyter_lsp | extension was successfully 

[BLAH BLAH BLAH]

[C 2025-11-04 13:00:03.841 ServerApp]

    To access the server, open this file in a browser:
        file:///home/hd/hd_hd/hd_cu284/.local/share/jupyter/runtime/jpserver-3805743-open.html
    Or copy and paste one of these URLs:
        http://172.0.0.1:18205/lab?token=e0d4bf618baa5d52cc7231c84922b6f19b961a167f8c89e2
        http://127.0.0.1:18205/lab?token=e0d4bf618baa5d52cc7231c84922b6f19b961a167f8c89e2
[I 2025-11-04 13:00:03.880 ServerApp] Skipped non-installed server(s): bash-language-server, dockerfile-language-server-nodejs, javascript-typesc>

Now run the ssh command (`ssh -N -L 18205:172.0.0.1:18205 hd_cu284@login.binac2.uni-tuebingen.de`) into a local cmd line and enter the password. If there is no (error) message, then that means it worked!

Now paste the second url (`http://127.0.0.1:18205/lab?token=e0d4bf618baa5d52cc7231c84922b6f19b961a167f8c89e2`) into your browser. The jupyter lab should now be ready.
(The first url doesn't work; I have no idea why.)

Note that the root of the jupyter server is the $HOME dir, so the one that is backed up regularly. Data from submitted jobs in work are somewhere else.

## Installing packages for the Jupyter Notebooks on BinAC

First, we have to activate miniforge so we can use conda.

`module load devel/miniforge/24.9.2`

(The section telling you that you have to install miniconda yourself is wrong. Not that I managed to install miniconda anyway :/ )

Now we can create a kernel follwing the instructions here: https://wiki.bwhpc.de/e/BinAC/Software/Jupyterlab
Once that's done, we can install new packages in the venv with the usual pip commands.

To use this new kernel, use the option `Python 3.8 (pandas)` when creating a new file.

The name of the new venv is `kernel_env`. It can be activated by running `conda activate kernel_env`. If ever in doubt about the name/existence of venvs, run `conda info --envs`.

## Still unclear
- What does the wiki page want me to do in **setting up a working directory**? There is no /beegfs/work/ pre constructed in the $HOME. 
- How to submit a job
- How to run a jupyter notebook
- What is going on with the storage system

WORK=/pfs/10/work/hd_cu284-work/
cd $WORK


to switch quickly.